# ML Ensemble Lab — Bagging vs Boosting
**Dataset:** `data2/heart_disease.csv` (10k × 21)
Objetivo: comparar **Bagging (RandomForest)** vs **Boosting (GradientBoosting / AdaBoost)** en **Clasificación** y **Regresión**.
- Clasificación: `Heart Disease Status` (Yes/No, desbalance 80/20)
- Regresión: `Cholesterol Level` (continuo) + alternativa `BMI`


In [ ]:
import os, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, RocCurveDisplay, mean_absolute_error, mean_squared_error,
                             r2_score, precision_recall_curve, average_precision_score, balanced_accuracy_score)
from sklearn.inspection import permutation_importance
import warnings; warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"]=120
DATA_PATH="../data2/heart_disease.csv"
if not os.path.exists(DATA_PATH):
    DATA_PATH="data2/heart_disease.csv"
print(DATA_PATH, os.path.exists(DATA_PATH))

## 1. Exploración del Dataset

In [ ]:
df=pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum().sort_values(ascending=False).head(10)

In [ ]:
df["Heart Disease Status"].value_counts(normalize=True)

In [ ]:
df[ ["Age","Blood Pressure","Cholesterol Level","BMI","Sleep Hours","Triglyceride Level"]].describe()

In [ ]:
plt.figure(figsize=(8,5))
sns.heatmap(df.select_dtypes(include=np.number).corr(), cmap="coolwarm", annot=False)
plt.title("Correlación — features numéricas")
plt.show()

In [ ]:
for c in df.select_dtypes(include="object").columns:
    print(c, ":", df[c].nunique(), "->", df[c].unique()[:4])

## 2. Preprocesamiento
- Numéricas: imputación mediana + StandardScaler
- Categóricas: imputación moda + OneHotEncoder

In [ ]:
TARGET_CLASS="Heart Disease Status"
TARGET_REG="Cholesterol Level"

def build_preprocessor(df, target_col):
    all_cols=[c for c in df.columns if c!=target_col]
    num_cols=df[all_cols].select_dtypes(include=np.number).columns.tolist()
    if target_col in num_cols: num_cols.remove(target_col)
    cat_cols=[c for c in all_cols if c not in num_cols]
    num_pipe=Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    cat_pipe=Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    pre=ColumnTransformer([("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)])
    return pre, num_cols, cat_cols

df_tmp=df.copy()
df_tmp[TARGET_CLASS]=df_tmp[TARGET_CLASS].map({"Yes":1, "No":0})
X=df_tmp.drop(columns=[TARGET_CLASS]); y=df_tmp[TARGET_CLASS]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2, random_state=42, stratify=y)
pre,nc,cc=build_preprocessor(pd.concat([X_train,X_test]), TARGET_CLASS)
print("Num:",nc[:5]); print("Cat:",cc[:5])

## 3. Clasificación — Heart Disease Status
Comparativa Bagging (RandomForest) vs Boosting (GradientBoosting, AdaBoost)
### Métricas + Gráficos prioritarios para dataset desbalanceado

In [ ]:
def run_classification(n_estimators=100, lr=0.1):
    df2=pd.read_csv(DATA_PATH)
    df2[TARGET_CLASS]=df2[TARGET_CLASS].map({"Yes":1, "No":0})
    df2=df2.dropna(subset=[TARGET_CLASS])
    X=df2.drop(columns=[TARGET_CLASS]); y=df2[TARGET_CLASS]
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
    pre,nc,cc=build_preprocessor(pd.concat([Xtr,Xte]), TARGET_CLASS)
    models={
        "Bagging_RF": RandomForestClassifier(n_estimators=n_estimators, random_state=42, n_jobs=-1, class_weight="balanced"),
        "Boosting_GB": GradientBoostingClassifier(n_estimators=n_estimators, learning_rate=lr, random_state=42),
        "Boosting_Ada": AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), n_estimators=n_estimators, learning_rate=lr, random_state=42)
    }
    res={}
    for name, clf in models.items():
        pipe=Pipeline([("pre", pre), ("clf", clf)])
        pipe.fit(Xtr,ytr)
        yp=pipe.predict(Xte)
        yp_proba=pipe.predict_proba(Xte)[:,1]
        res[name]={"pipe":pipe,"yp":yp,"proba":yp_proba,
            "acc":accuracy_score(yte,yp),"prec":precision_score(yte,yp,zero_division=0),
            "rec":recall_score(yte,yp,zero_division=0),"f1":f1_score(yte,yp,zero_division=0),
            "roc":roc_auc_score(yte,yp_proba),"yte":yte}
    metrics=pd.DataFrame({k:{m:v for m,v in r.items() if m in ["acc","prec","rec","f1","roc"]} for k,r in res.items()}).T
    display(metrics.round(3))
    
    # 1) ROC Curves
    plt.figure(figsize=(5,4))
    for n,r in res.items():
        RocCurveDisplay.from_predictions(r["yte"], r["proba"], name=n, ax=plt.gca())
    plt.plot([0,1],[0,1],"k--",lw=0.8); plt.title("ROC — Bagging vs Boosting"); plt.show()
    
    # 2) Matrices de confusión
    fig, ax=plt.subplots(1,3, figsize=(12,3.5))
    for i,(n,r) in enumerate(res.items()):
        sns.heatmap(confusion_matrix(r["yte"], r["yp"]), annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax[i])
        ax[i].set_title(n); ax[i].set_xlabel("Pred"); ax[i].set_ylabel("Real")
    plt.tight_layout(); plt.show()
    
    # 3) Precision-Recall (mejor que ROC para clase minoritaria)
    plt.figure(figsize=(5,4))
    for n,r in res.items():
        prec, rec, _ = precision_recall_curve(r["yte"], r["proba"])
        ap = average_precision_score(r["yte"], r["proba"])
        plt.plot(rec, prec, label=f"{n} (AP={ap:.3f})", lw=2)
    baseline = res[list(res.keys())[0]]["yte"].mean()
    plt.axhline(baseline, color="gray", ls="--", lw=1, label=f"Baseline ({baseline:.3f})")
    plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision-Recall"); plt.legend(); plt.show()
    
    # 4) Distribución de probabilidades (diagnostica AUC ~0.5)
    for n,r in res.items():
        plt.figure(figsize=(5,4))
        sns.kdeplot(x=r["proba"][r["yte"]==0], fill=True, alpha=0.35, label="Clase 0", color="#4575b4")
        sns.kdeplot(x=r["proba"][r["yte"]==1], fill=True, alpha=0.35, label="Clase 1", color="#d73027")
        plt.axvline(0.5, color="black", ls="--", label="Umbral 0.5")
        plt.xlabel("Probabilidad predicha de Clase 1"); plt.title(f"Probabilidades - {n}"); plt.legend(); plt.show()
    
    # 5) Métricas vs Umbral (elige corte óptimo)
    thresholds = np.arange(0.05, 0.96, 0.05)
    plt.figure(figsize=(6,4))
    for n,r in res.items():
        precs, recs, f1s = [], [], []
        for th in thresholds:
            y_pred_th = (r["proba"] >= th).astype(int)
            precs.append(precision_score(r["yte"], y_pred_th, zero_division=0))
            recs.append(recall_score(r["yte"], y_pred_th, zero_division=0))
            f1s.append(f1_score(r["yte"], y_pred_th, zero_division=0))
        plt.plot(thresholds, precs, '--', label=f"{n} Precision", alpha=0.7)
        plt.plot(thresholds, recs, '-.', label=f"{n} Recall", alpha=0.7)
        plt.plot(thresholds, f1s, '-', label=f"{n} F1", lw=2)
    plt.axvline(0.5, color="gray", ls=":", label="Umbral 0.5")
    plt.xlabel("Umbral"); plt.ylabel("Métrica"); plt.title("Métricas vs Umbral"); plt.legend(ncol=2); plt.grid(alpha=0.3); plt.show()
    
    # 6) Importancia por permutación (más robusta)
    for n in ["Bagging_RF", "Boosting_GB"]:
        pipe=res[n]["pipe"]
        result = permutation_importance(pipe, Xte, yte, scoring="average_precision", n_repeats=10, random_state=42, n_jobs=-1)
        cat_enc=pipe.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"]
        feat_names=nc + cat_enc.get_feature_names_out(cc).tolist()
        imp_mean = result.importances_mean
        idx = np.argsort(imp_mean)[::-1][:15]
        plt.figure(figsize=(6,5))
        plt.barh(range(len(idx)), imp_mean[idx[::-1]], color="teal")
        plt.yticks(range(len(idx)), [feat_names[i] for i in idx[::-1]], fontsize=9)
        plt.xlabel("Disminución media en Average Precision")
        plt.title(f"Permutation Importance - {n}"); plt.tight_layout(); plt.show()
    
    # 7) Feature importance nativa (top 10)
    for n in ["Bagging_RF","Boosting_GB"]:
        pipe=res[n]["pipe"]; pre_step=pipe.named_steps["pre"]
        cat_enc=pre_step.named_transformers_["cat"].named_steps["ohe"]
        feat_names=nc + cat_enc.get_feature_names_out(cc).tolist()
        imp=pipe.named_steps["clf"].feature_importances_
        idx=np.argsort(imp)[::-1][:10]
        plt.figure(figsize=(6,3))
        sns.barplot(x=imp[idx], y=[feat_names[i] for i in idx], hue=[feat_names[i] for i in idx], legend=False)
        plt.title(f"Importancia nativa — {n}"); plt.tight_layout(); plt.show()
    
    return res, Xte, yte

res_cls, X_test_cls, y_test_cls = run_classification(n_estimators=100, lr=0.1)

> **Interpretación:** Dataset sintético ruidoso → AUC ~0.5. Se observa: **Bagging** estabiliza (reduce varianza) y **Boosting** intenta corregir sesgo pero puede sobreajustar con `learning_rate` alto. Con datos reales la brecha sería mayor.

## 4. Regresión — Cholesterol Level (y alternativa BMI)
Comparativa Bagging (RF Regressor) vs Boosting (GB Regressor)

In [ ]:
def run_regression(target="Cholesterol Level", n_estimators=100, lr=0.1):
    df2=pd.read_csv(DATA_PATH).dropna(subset=[target])
    drop_cols=[target]
    if TARGET_CLASS in df2.columns: drop_cols.append(TARGET_CLASS)
    X=df2.drop(columns=drop_cols); y=df2[target]
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,random_state=42)
    pre,nc,cc=build_preprocessor(pd.concat([Xtr,Xte]), target)
    models={
        "Bagging_RF": RandomForestRegressor(n_estimators=n_estimators, random_state=42, n_jobs=-1),
        "Boosting_GB": GradientBoostingRegressor(n_estimators=n_estimators, learning_rate=lr, random_state=42)
    }
    res={}
    for n,reg in models.items():
        pipe=Pipeline([("pre", pre), ("reg", reg)])
        pipe.fit(Xtr,ytr)
        yp=pipe.predict(Xte)
        res[n]={"pipe":pipe,"yp":yp,"yte":yte,
                "mae":mean_absolute_error(yte,yp),"mse":mean_squared_error(yte,yp),
                "rmse":np.sqrt(mean_squared_error(yte,yp)),"r2":r2_score(yte,yp)}
    metrics=pd.DataFrame({k:{m:float(v) for m,v in r.items() if m in ["mae","mse","rmse","r2"]} for k,r in res.items()}).T
    display(metrics.round(3))
    # Scatter Real vs Pred
    fig, ax=plt.subplots(1,2, figsize=(10,4))
    for i,(n,r) in enumerate(res.items()):
        ax[i].scatter(r["yte"], r["yp"], alpha=0.3, s=10)
        mn=min(r["yte"].min(), r["yp"].min()); mx=max(r["yte"].max(), r["yp"].max())
        ax[i].plot([mn,mx],[mn,mx],"r--"); ax[i].set_title(n); ax[i].set_xlabel("Real"); ax[i].set_ylabel("Pred")
    plt.suptitle(f"Real vs Predicho — {target}"); plt.tight_layout(); plt.show()
    # Residuales
    fig, ax=plt.subplots(1,2, figsize=(10,4))
    for i,(n,r) in enumerate(res.items()):
        ax[i].scatter(r["yp"], r["yte"]-r["yp"], alpha=0.3, s=10, color="darkred")
        ax[i].axhline(0, color="black", ls="--"); ax[i].set_title(n+" — Residuales") ; ax[i].set_xlabel("Predicho"); ax[i].set_ylabel("Residual")
    plt.tight_layout(); plt.show()
    # Feature importance
    for n in res:
        pipe=res[n]["pipe"]; pre_step=pipe.named_steps["pre"]
        cat_enc=pre_step.named_transformers_["cat"].named_steps["ohe"]
        feat_names=nc + cat_enc.get_feature_names_out(cc).tolist()
        imp=pipe.named_steps["reg"].feature_importances_
        idx=np.argsort(imp)[::-1][:10]
        plt.figure(figsize=(6,3))
        sns.barplot(x=imp[idx], y=[feat_names[i] for i in idx], hue=[feat_names[i] for i in idx], legend=False)
        plt.title(f"Importancia — {n} ({target})"); plt.tight_layout(); plt.show()
    return res

res_reg=run_regression("Cholesterol Level", n_estimators=100, lr=0.1)
print("--- Alternativa BMI ---")
res_reg_bmi=run_regression("BMI", n_estimators=100, lr=0.1)

## 5. Comparativa Final — Tabla Resumen

In [ ]:
print("Clasificación — F1 / ROC-AUC (Bagging vs Boosting)")
for k,v in res_cls.items():
    print(k, f"F1={v['f1']:.3f} ROC={v['roc']:.3f}")
print("\nRegresión Cholesterol — R2 / RMSE")
for k,v in res_reg.items():
    print(k, f"R2={v['r2']:.3f} RMSE={v['rmse']:.1f}")
print("\nRegresión BMI — R2 / RMSE")
for k,v in res_reg_bmi.items():
    print(k, f"R2={v['r2']:.3f} RMSE={v['rmse']:.2f}")